In [ ]:
!pip install boto3 botocore

In [2]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import random

# =========================================================
# CONFIGURAÇÕES
# =========================================================

bucket = "physionet-open"
base_prefix = "eegmmidb/1.0.0/"
local_root = "data_eegmmidb"

# Quantidade de pacientes que deseja baixar
NUM_PATIENTS = 10

# Seed fixa para garantir reprodutibilidade
# (todos que usarem essa mesma seed e mesmo NUM_PATIENTS
# terão exatamente os mesmos pacientes sorteados)
SEED = 42

# Total de sujeitos disponíveis no dataset: S001 até S109
TOTAL_PATIENTS = 109

# =========================================================
# PREPARAÇÃO
# =========================================================

os.makedirs(local_root, exist_ok=True)

s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED)
)

# =========================================================
# SORTEIO REPRODUTÍVEL DOS PACIENTES
# =========================================================

all_patients = [f"S{i:03d}" for i in range(1, TOTAL_PATIENTS + 1)]

random.seed(SEED)
selected_patients = sorted(random.sample(all_patients, NUM_PATIENTS))

print("Pacientes selecionados:")
print(selected_patients)

# =========================================================
# DOWNLOAD DOS PACIENTES SELECIONADOS
# =========================================================

for patient in selected_patients:
    prefix = f"{base_prefix}{patient}/"
    local_dir = os.path.join(local_root, patient)
    os.makedirs(local_dir, exist_ok=True)

    print(f"\n=== Verificando {patient} ===")

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]

            if key.endswith("/"):
                continue

            local_path = os.path.join(
                local_dir,
                os.path.basename(key)
            )

            if os.path.exists(local_path):
                print("Skipping (já existe):", local_path)
                continue

            print("Downloading", key)
            s3.download_file(bucket, key, local_path)

print("\nDownload concluído.")

Pacientes selecionados:
['S004', 'S014', 'S015', 'S018', 'S029', 'S032', 'S036', 'S082', 'S087', 'S095']

=== Verificando S004 ===
Skipping (já existe): data_eegmmidb\S004\S004R01.edf
Skipping (já existe): data_eegmmidb\S004\S004R01.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R02.edf
Skipping (já existe): data_eegmmidb\S004\S004R02.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R03.edf
Skipping (já existe): data_eegmmidb\S004\S004R03.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R04.edf
Skipping (já existe): data_eegmmidb\S004\S004R04.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R05.edf
Skipping (já existe): data_eegmmidb\S004\S004R05.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R06.edf
Skipping (já existe): data_eegmmidb\S004\S004R06.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R07.edf
Skipping (já existe): data_eegmmidb\S004\S004R07.edf.event
Skipping (já existe): data_eegmmidb\S004\S004R08.edf
Skipping (já existe): data_eegmm